## Depenedencies and OpenAI API Key

We'll be using OpenAI's suite of models today to help us generate and embed our documents for a simple RAG system built on top of LangChain's blogs!

In [1]:
%%capture
!pip install -U openai pydantic langchain langchain-community langchain-openai langchainhub langsmith  tiktoken cohere -qU

In [2]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

Enter your OpenAI API Key:··········


## Basic RAG Chain

Now we'll set up our basic RAG chain, first up we need a model!

### OpenAI Model


We'll use OpenAI's `gpt-3.5-turbo` model to ensure we can use a stronger model for decent evaluation later!

Notice that we can tag our resources - this will help us be able to keep track of which resources were used where later on!

In [3]:
from langchain_openai import ChatOpenAI

base_llm = ChatOpenAI(model="gpt-3.5-turbo", tags=["base_llm"])

#### Asyncio Bug Handling

This is necessary for Colab.

In [4]:
import nest_asyncio
nest_asyncio.apply()

### SiteMap Loader

We'll use a SiteMapLoader to scrape the LangChain blogs.

In [48]:
import requests
from bs4 import BeautifulSoup
from langchain_core.documents import Document

# 1. Fetch the sitemap
resp = requests.get("https://blog.langchain.dev/sitemap-posts.xml")
resp.raise_for_status()

# 2. Parse XML to extract URLs
soup = BeautifulSoup(resp.text, "xml")
urls = [loc.text for loc in soup.find_all("loc")][:5]

print(f"Processing {len(urls)} URLs...")

# 3. Define custom loader function
def fetch_and_parse(url):
    try:
        html = requests.get(url, timeout=10).text
        page_soup = BeautifulSoup(html, "html.parser")
        article = page_soup.find("article") or page_soup.body
        text = article.get_text(separator="\n", strip=True) if article else "No content found"
        return Document(page_content=text, metadata={"source": url})
    except Exception as e:
        print(f"Failed to fetch {url}: {e}")
        return None

# 4. Load documents
documents = [doc for url in urls if (doc := fetch_and_parse(url))]

print(f"Fetched {len(documents)} documents successfully.")

Processing 5 URLs...
Fetched 5 documents successfully.


In [52]:
documents[0]

Document(metadata={'source': 'https://blog.langchain.com/introducing-open-swe-an-open-source-asynchronous-coding-agent/'}, page_content="Introducing Open SWE: An Open-Source Asynchronous Coding Agent\n7 min read\nAug 6, 2025\nThe use of AI in software engineering has evolved over the past two years. It started as autocomplete, then went to a copilot in an IDE, and in the fast few months has evolved to be a long running, more end-to-end agent that run asynchronously in the cloud.\nWe believe that all agents will long more like this in the future - long running, asynchronous, more autonomous. Specifically, we think that they will:\nRun asynchronously in the cloud\nIntegrate directly with your tooling\nHave enough context over your environment to properly plan tasks over longer time horizons\nReview their own work (and fix any issues) before completing their task\nOver the past few months it became apparent that software engineering was the first discipline where this vision would become 

In [49]:
documents[0].metadata["source"]

'https://blog.langchain.com/introducing-open-swe-an-open-source-asynchronous-coding-agent/'

### RecursiveCharacterTextSplitter

We're going to use a relatively naive text splitting strategy today!

In [53]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

split_documents = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size = 256,
    chunk_overlap = 16
).split_documents(documents)

In [54]:
len(split_documents)

24

In [55]:
split_documents[2]

Document(metadata={'source': 'https://blog.langchain.com/introducing-open-swe-an-open-source-asynchronous-coding-agent/'}, page_content="examples page in our documentation\n.\nWhy Open SWE?\nThere are a number of open source coding projects. Why built a new one? We wanted to drive attention and focus to aspects besides just the prompts and tools used. Specifically, we wanted to highlight more of the overall flow and UX that is needed to bring these agents to the point where we can interact with them in a reliable way.\nWe think UI/UX is often the one of the more under-explored areas in agent building. The overall interaction pattern of your application can greatly determine the usage it gets. With asynchronous agents being such a new idea, we think there are a lot of interesting patterns to explore here. Two main points are\nmore control\nand\ndeep integration\n.\nControl:\nOpen SWE has two main features that give you more control over your coding agent while it's running. You can inte

### Embeddings

We'll be leveraging OpenAI's Embeddings Models today!

In [28]:
from langchain_openai import OpenAIEmbeddings

base_embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")

### FAISS VectorStore Retriever

Now we can use a FAISS VectorStore to embed and store our documents and then convert it to a retriever so it can be used in our chain!

In [29]:
%%capture
!pip install faiss-cpu -qU

In [30]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(split_documents, base_embeddings_model)

In [31]:
base_retriever = vectorstore.as_retriever()

### Prompt Template

All we have left is a prompt template, which we'll create here!

In [32]:
from langchain.prompts import ChatPromptTemplate

base_rag_prompt_template = """\
Using the provided context, please answer the user's question. If you don't know the answer based on the context, say you don't know.

Context:
{context}

Question:
{question}
"""

base_rag_prompt = ChatPromptTemplate.from_template(base_rag_prompt_template)

### LCEL Chain

Now that we have:

- Embeddings Model
- Generation Model
- Retriever
- Prompt

We're ready to build our LCEL chain!

Keep in mind that we're returning our source documents with our queries - while this isn't necessary, it's a great thing to get into the habit of doing.

In [33]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain.schema import StrOutputParser

base_rag_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | base_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": base_rag_prompt | base_llm | StrOutputParser(), "context": itemgetter("context")}
)

Let's test it out!

In [56]:
base_rag_chain.invoke({"question" : "What is a good way to evaluate agents?"})

{'response': "A good way to evaluate agents can be through performance metrics such as completion rates, accuracy, efficiency, and customer satisfaction. Another method could be through A/B testing or simulations to measure the agent's effectiveness in various scenarios.",
 'context': [Document(id='718d6702-5389-41c7-88e0-805dc52f6b44', metadata={'source': 'https://blog.langchain.dev/langgraph-intro/', 'title': 'LangChain Blog', 'language': 'en'}, page_content='LangChain Blog\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nSkip to content\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nCase Studies\n\n\n\n\nIn the Loop\n\n\n\n\nLangChain\n\n\n\n\nDocs\n\n\n\n\nChangelog\n\n\n\n\n\nSign in\nSubscribe\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n404\nPage not found\nGo to the homepage\n\n\n\n\n\n\n\n\nSign up\n\n\n\n\n\n            Â© LangChain Blog 2025'),
  Document(id='204edc31-d57b-4dd9-8f3e-4d719e21a154', metadata={'source': 'https://blog.langchain.dev/agentic-rag/', 'title': 'LangChain Blog', 'language': 'e

## LangSmith

Now that we have a chain - we're ready to get started with LangSmith!

We're going to go ahead and use the following `env` variables to get our Colab notebook set up to start reporting.

If all you needed was simple monitoring - this is all you would need to do!

In [57]:
from uuid import uuid4

unique_id = uuid4().hex[0:8]

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = f"Langsmith_RAG_{unique_id}"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"

### LangSmith API


In [58]:
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass('Enter your LangSmith API key: ')

Enter your LangSmith API key: ··········


Let's test our our first generation!

In [59]:
base_rag_chain.invoke({"question" : "What is LangSmith?"}, {"tags" : ["Demo Run"]})['response']

"I don't know."

## Create Testing Dataset

Now we can create a dataset using some user defined questions, and providing the retrieved context as a "ground truth" context.

> NOTE: There are many different ways you can approach this specific task - generating ground truth answers with AI, using human experts to generate golden datasets, and more!

In [38]:
from langsmith import Client

test_inputs = [
    "What is LangSmith?",
    "What is LangServe?",
    "How could I benchmark RAG on tables?",
    "What was exciting about LangChain's first birthday?",
    "What features were released for LangChain on August 7th?",
    "What is a conversational retrieval agent?"
]

client = Client()

dataset_name = "langsmith-demo-dataset-v1"

dataset = client.create_dataset(
    dataset_name=dataset_name, description="LangChain Blog Test Questions"
)

for input in test_inputs:
  client.create_example(
      inputs={"question" : input},
      outputs={"answer" : base_rag_chain.invoke({"question" : input})["context"]},
      dataset_id=dataset.id
  )

### Evaluation

Now we can run the evaluation!

In [51]:
from langchain.smith import RunEvalConfig, run_on_dataset

eval_llm = ChatOpenAI(model="gpt-4-0125-preview", temperature=0)

eval_config = RunEvalConfig(
  evaluators=[
    RunEvalConfig.CoTQA(llm=eval_llm, prediction_key="response"),
    RunEvalConfig.Criteria("harmfulness", prediction_key="response"),
  ]
)

base_rag_base_run = run_on_dataset(
    client=client,
    dataset_name=dataset_name,
    llm_or_chain_factory=base_rag_chain,
    evaluation=eval_config,
    verbose=True,
)

View the evaluation results for project 'ample-wire-92' at:
https://smith.langchain.com/o/efe14579-4a50-4493-9f1f-f21074492a30/datasets/527b6c51-3d12-4a69-8553-b25936bf8abf/compare?selectedSessions=2ebae54b-00e2-4e26-94ba-e9f338228968

View all tests for Dataset langsmith-demo-dataset-v1 at:
https://smith.langchain.com/o/efe14579-4a50-4493-9f1f-f21074492a30/datasets/527b6c51-3d12-4a69-8553-b25936bf8abf
[------------------------------------------------->] 6/6

 Experiment Results:
        feedback.Contextual Accuracy  feedback.harmfulness error  execution_time                                run_id
count                           6.00                  6.00     0            6.00                                     6
unique                           NaN                   NaN     0             NaN                                     6
top                              NaN                   NaN   NaN             NaN  cf8486f3-9b38-439b-a765-f86766daeb05
freq                             NaN   

## Adding Reranking

We'll add reranking to our RAG application to confirm the claim made by [Cohere](https://cohere.com/rerank)!

`Improve search performance with a single line of code`

We'll put that to the test today!

In [ ]:
os.environ["COHERE_API_KEY"] = getpass.getpass("Enter your Cohere API Key:")

In [ ]:
base_retriever_expander = vectorstore.as_retriever(
    search_kwargs={"k" : 10}
)

In [ ]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import CohereRerank

reranker = CohereRerank()
rerank_retriever = ContextualCompressionRetriever(
    base_compressor=reranker, base_retriever=base_retriever_expander
)

### Recreating our Chain with Reranker

Now we can recreate our chain using the reranker.

In [ ]:
rerank_rag_chain = (
    {"context": itemgetter("question") | rerank_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": base_rag_prompt | base_llm | StrOutputParser(), "context": itemgetter("context")}
)

rerank_rag_chain = rerank_rag_chain.with_config({"tags" : ["cohere-rerank"]})

### Improved Evaluation

Now we can leverage the full suite of LangSmith's evaluation to evaluate our chains on multiple metrics, including custom metrics!

In [ ]:
eval_config = RunEvalConfig(
  evaluators=[
    RunEvalConfig.CoTQA(llm=eval_llm, prediction_key="response"),
    RunEvalConfig.Criteria("harmfulness", prediction_key="response"),
    RunEvalConfig.LabeledCriteria(
        {
            "helpfulness" : (
                "Is this submission helpful to the user,"
                "taking into account the correct reference answer?"
            )
        },
        prediction_key="response"
    ),
    RunEvalConfig.LabeledCriteria(
        {
            "litness" : (
                "Is this submission lit, dope, or cool?"
            )
        },
        prediction_key="response"
    ),
    RunEvalConfig.LabeledCriteria("conciseness", prediction_key="response"),
    RunEvalConfig.LabeledCriteria("coherence", prediction_key="response"),
    RunEvalConfig.LabeledCriteria("relevance", prediction_key="response")
  ]
)

### Running Eval on Each Chain

Now we can evaluate each of our chains!

In [ ]:
base_chain_results = run_on_dataset(
    client=client,
    dataset_name=dataset_name,
    llm_or_chain_factory=base_rag_chain,
    evaluation=eval_config,
    verbose=True,
)

In [ ]:
rerank_chain_results = run_on_dataset(
    client=client,
    dataset_name=dataset_name,
    llm_or_chain_factory=rerank_rag_chain,
    evaluation=eval_config,
    verbose=True,
)